# Install requirements and dependencies

In [ ]:
PROJECT_PATH = "/home/jupyter-gabriel/projects/pricing_automation/pricing-automation/etl_pipeline"

In [ ]:
%pip install -r "$PROJECT_PATH/requirements.txt"

## Load kedro with magic command

In [ ]:
%load_ext kedro.ipython

Change the notebook to the project path and reload the project from there

In [ ]:
%cd $PROJECT_PATH
%reload_kedro .

## (Optional) Load custom functions

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pricing_automation.pipelines.utils import *
from pricing_automation.pipelines.a01_aoi_period import *
from pricing_automation.pipelines.n01_extract_data import *
from pricing_automation.pipelines.n02_process_data import *
from pricing_automation.pipelines.n03_create_triggers import *

# Create a Session

In [ ]:
%reload_kedro
runtime_params = {
    'country': 'argentina', # Local folder name to store data 
    'region': 'buenos_aires',
    'lead_id': 'test', # Lead name
    'loc_id': 'SC-canton-043',
    'provider': 'UCSB', # Data provider, can be 'ERA5' or 'UCSB'
    'field': 'tmin', # Short variable name, can be 'swc', 'prcp', 'tmin', 'tmax'
    'start_year': 2010, # (optional) Use this to override the initial year of data
    'end_year': 2025, # (optional) Use this to override the ending year of data
}
session_tmin = session.create(
    env='tmin',
    runtime_params = runtime_params
)
# (Optional) Load the catalog with the context provided earlier for quick loads
catalog_tmin = session_tmin.load_context().catalog 

# Step 0. Upload geometry

In [ ]:
#OPTIONAL If you are able to provide with a context file for geometry and boundaries upload it so that the Kedro project can use it
#gdf_ar_depto = gpd.read_file('/home/jupyter-gabriel/suyana/geometries/argentina/AR_departamento.zip')
#gdf_ba_depto = gdf_ar_depto[gdf_ar_depto['fdc']=='ARBA - Gerencia de Servicios Catastrales'].copy()
#catalog_tmin.save('gdf_request', gdf_ba_depto) # This uploads the file gdf_ba_depto as the 'gdf_request' entry in the catalog.yml

In [ ]:
gdf_request = catalog_tmin.load('gdf_request')

# Step 1. Download data

In [ ]:
#———————————————————————————————
# NOTE
# env = <env_name> parameter in session.create() tells Kedro to 
# use the files in the conf/<env_name> as the main .yml files, 
# but reverting back to those in conf/base if they are not found in conf/<env_name>
# if they are found in both it merges them prioritizing the specified conf/<env_name>
#————————————————————————————————
#———————————————————————————————
# NOTE
# If the time it takes to download full years takes more than an hour
# It is better to split the downloads by year and save the results partially
# To do so make a loop to run the the nodes for extracting data
# by iterating on runtime_params['start_year'] and runtime_params['end_year']
#————————————————————————————————
%reload_kedro
session.create(
    env = 'tmin',
    runtime_params = runtime_params
).run(
    node_names = ['get_aoi', 'extract_data']
)
'''
Iterating to store years partially
sy, ey = 1996, 2025
list_years = np.arange(sy, ey+1).tolist()
for year in list_years:
    runtime_params['start_year'] = year
    runtime_params['end_year'] = year
    session.create(
        env = 'tmin',
        runtime_params = runtime_params
    ).run(
        node_names = ['get_aoi', 'extract_data']
    )

# Restore values
runtime_params['start_year'] = sy
runtime_params['end_year'] = ey

session_tmin = session.create(
    env='tmin',
    runtime_params = runtime_params
)
catalog_tmin = session_tmin.load_context().catalog 
'''

# Step 2: Process data

In [ ]:
%reload_kedro
session.create(
    env='tmin',
    runtime_params = runtime_params, 
).run(
    node_names = ['process_data']
)

# Step 3. Summarize Data

In [ ]:
%reload_kedro
session.create(
    env='tmin',
    runtime_params = runtime_params,
).run(
    node_names = ['summarize_processed_data']
)

# Step 4. Payouts and premiums

In [ ]:
%reload_kedro
session.create(
    env='tmin',
    runtime_params = runtime_params,
).run(
    #node_names = ['summarize_processed_data']
    tags = ['triggers']
)

# Step 5. Bootstrap aep

In [ ]:
%reload_kedro
session.create(
    env='tmin',
    runtime_params = runtime_params
).run(
    tags=['aep']
)

# Step 6. Create auxiliary plots

In [ ]:
%reload_kedro
session.create(
    env='tmin',
    runtime_params = runtime_params
).run(
    nodes=['plot_trigger_frequency_map', 'create_activation_map']
)